In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

import torch

In [3]:
dali_directory = Path('../alignment_results/Llama3.2-1B/belebele/dali')
tlang = [lang.name.split("_")[1] for lang in dali_directory.glob("*.json")]
def get_max_lambda(metric):
    files = sorted(Path(f"../alignment_results/Llama3.2-1B/belebele/{metric}").glob("*.json"))
    max_lamda = []

    for f in files:
        df = pd.read_json(f)
        if(metric =="mexa"):
            df = df.T
        percentages = {i: df.loc[i].mean() * 100 for i in range(17)}
        
        best_layer = max(percentages, key=percentages.get)
        # print(percentages)
        if percentages[best_layer] == 0:
            max_lamda.append(None)
        else:
            max_lamda.append(best_layer)
    return max_lamda

dali_max_lamda = get_max_lambda("dali")
dalist_max_lamda = get_max_lambda("dali_strong")
mexa_max_lamda = get_max_lambda("mexa")

belebele_max_lamda_df = pd.DataFrame({
    "target_languages": tlang,
    "dali_max_lamda": dali_max_lamda,
    "dalist_max_lamda": dalist_max_lamda,
    "mexa_max_lamda": mexa_max_lamda
})
print ("fodsjg")
belebele_max_lamda_df

fodsjg


,target_languages,dali_max_lamda,dalist_max_lamda,mexa_max_lamda
0,deu,10,6,5
1,fra,10,5,5
2,hin,7,4,8
3,ita,7,7,5
4,por,10,5,5
5,spa,10,5,5


In [12]:
eng_df = pd.read_json("../results/patching/Llama3.2-1B/xcopa/xcopa_eval_Llama3.2-1B_it_AB.jsonl", lines = True)
eng_df = eng_df[eng_df["acc"]==1]
eng_df

,idx,acc,selected_logits,correct_ans
1,1,1,"[11.1953125, 10.9765625]",0
3,3,1,"[11.6328125, 11.1328125]",0
8,8,1,"[11.8984375, 11.8125]",0
10,10,1,"[11.9296875, 11.3984375]",0
11,11,1,"[11.34375, 11.484375]",1
...,...,...,...,...
486,486,1,"[11.296875, 10.6796875]",0
487,487,1,"[12.1484375, 11.8515625]",0
490,490,1,"[11.578125, 11.328125]",0
493,493,1,"[11.8828125, 11.71875]",0


In [14]:
import pandas as pd

def get_xcopa_ts_tf(lang, num_samples=None):
    base = "../results/accuracy/Llama3.2-1B/xcopa"
    eng_df = pd.read_json(f"{base}/xcopa_eval_Llama3.2-1B_translation-{lang}_AB.jsonl", lines=True)
    lang_df = pd.read_json(f"{base}/xcopa_eval_Llama3.2-1B_{lang}_AB.jsonl", lines=True)

    if num_samples is not None:
        eng_df = eng_df[eng_df["idx"] < num_samples]
        lang_df = lang_df[lang_df["idx"] < num_samples]

    merged = eng_df[["idx", "acc"]].merge(
        lang_df[["idx", "acc"]],
        on="idx",
        suffixes=("_eng", "_lang")
    )

    ts_idx = merged.loc[(merged["acc_eng"] == 1.0) & (merged["acc_lang"] == 1.0), "idx"].tolist()
    tf_idx = merged.loc[(merged["acc_eng"] == 1.0) & (merged["acc_lang"] == 0.0), "idx"].tolist()

    print(f"{lang.upper()} | Eng Acc: {eng_df['acc'].mean()*100:.1f}% | Lang Acc: {lang_df['acc'].mean()*100:.1f}% | TS: {len(ts_idx)} | TF: {len(tf_idx)}")
    return ts_idx, tf_idx

it_ts, it_tf = get_xcopa_ts_tf("it")
th_ts, th_tf = get_xcopa_ts_tf("th")

IT | Eng Acc: 56.8% | Lang Acc: 50.8% | TS: 155 | TF: 129
TH | Eng Acc: 51.8% | Lang Acc: 50.0% | TS: 187 | TF: 72
